In [1]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
import yaml
from pathlib import Path
import os
import re
import matplotlib.pyplot as plt
from typing import Annotated, Sequence, TypedDict, Dict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool
from transformers import pipeline
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

/home/mohbakr/Documents/enviroments/pytolib/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-21 14:18:38.089101: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent

# Load config
with open(PARENT_DIR /  "config.yaml", "r") as file:
    config = yaml.safe_load(file)

api_key = config["GROQ_API_KEY"]
embedding_model_name = config["EMBEDDING_MODEL"]

# Initialize LLM - Using a standard Groq model ID
llm = ChatGroq(
    api_key=api_key, 
    model="llama-3.3-70b-versatile", 
    temperature=0
)

embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

In [8]:
pdf_path = "../pdfs/iesc104.pdf"
loader = PyPDFLoader(pdf_path).load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(loader)

# Fixed typo: vectoerDB -> vector_db
vector_db = Chroma.from_documents(texts, embeddings)
retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [12]:
res=retriever.invoke('what is The Structure of an Atom')
for r in res:
    print(r.page_content)
    print("===================")

constituents inside the atom? W e shall find
out the answers to these questions in this
chapter . W e will lear n about sub-atomic
particles and the various models that have
been proposed to explain how these particles
are arranged within the atom.
A major challenge before the scientists at
the end of the 19th century was to reveal the
structure of the atom as well as to explain its
important properties. The elucidation of the
structure of atoms is based on a series of
experiments.
STRUCTURE OF THE ATOM 43
Table 4.1: Composition of Atoms of the First Eighteen Elements
with Electron Distribution in Various Shells
helium atom has two electrons in its outermost
shell and all other elements have atoms with
eight electrons in the outermost shell.
The combining capacity of the atoms of
elements, that is, their tendency to react and
form molecules with atoms of the same or
different elements, was thus explained as an
attempt to attain a fully-filled outermost shell.
theory. It was then consid

In [13]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a factual QA assistant. Answer ONLY from the given context.

Rules:
- Do not use outside knowledge
- Do not guess
- If missing → say "Not found in context"

Context:
{context}

Question:
{question}

Answer:
"""
)

In [15]:
Rag_chain=prompt|llm|StrOutputParser()

In [ ]:
from langchain_classic.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

query = "what is the sturcture of the atom"
answer = qa_chain.invoke(query)

print(answer)

{'query': 'what is the sturcture of the atom', 'result': "According to Rutherford's model of the atom, the structure consists of: \n(i) A very tiny nucleus present inside the atom \n(ii) Electrons revolve around this nucleus."}
